<a href="https://colab.research.google.com/github/Hussain-Ed/OEL_CV/blob/main/OEL_Computer_Vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
# List to find your folder name
os.listdir('/content/drive/MyDrive/')

['Colab Notebooks',
 '23-Ai-78_IS_oel2.pdf',
 'Classroom',
 'TV Series',
 'How To Get Crash Dump.zip (Unzipped Files)',
 'Untitled document.gdoc',
 'Untitled spreadsheet (1).gsheet',
 'HUSSAIN SHAKOOR .pdf',
 'Hussain-Shakoor (1).pdf',
 'Hussain-Shakoor.pdf',
 'Digital twins in Theory and Practice.pdf',
 'Deep classrooms digital twins.pdf',
 'Building metaverse using digital twins.pdf',
 'Applying Digital Twins in Metaverse.pdf',
 'Current trends in Digital twins 2024.pdf',
 'DIGITAL_TWIN_AND_BLOCKCHAIN.pdf',
 'DIGITAL_TWIN_BIOMEDICINE.pdf',
 'DIGITAL_TWIN_FOR_5G_AND_BEYOND.pdf',
 'DIGITAL_TWIN_FOR_DISASTER_MANAGEMENT.pdf',
 'DIGITAL_TWIN_INDUSTRY_4.0.pdf',
 'SMART_CITY_DIGITAL_TWIN.pdf',
 'Untitled spreadsheet.gsheet',
 "Here's the elaborated table with each aspect desc....gsheet",
 'articles',
 'Resume_02.pdf',
 'lab-7 (1).gdoc',
 'lab-7.gdoc',
 '23-Ai-78.pdf',
 'IMG_20250603_201747_811.jpg',
 'PE Assignment_78 (1).pdf',
 'PE Assignment_78.pdf',
 'lct assignment_78 (1).pdf',
 'lct as

In [3]:
DATA_DIR = '/content/drive/MyDrive/archive.zip'

In [4]:
import os, cv2, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.callbacks import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [6]:
DATA_DIR   = "PlantVillage"
IMG_SIZE   = 128
BATCH      = 32
EPOCHS_NN  = 10
EPOCHS_CNN = 15
EPOCHS_MN  = 10

In [8]:
def load_images(data_dir, img_size=IMG_SIZE):
    images, labels = [], []
    classes = sorted(os.listdir(data_dir))
    for cls in classes:
        cls_path = os.path.join(data_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        for fname in os.listdir(cls_path):
            path = os.path.join(cls_path, fname)
            img  = cv2.imread(path)
            if img is None:
                continue

            # Grayscale + histogram equalization
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            gray = cv2.equalizeHist(gray)

            # Resize
            gray = cv2.resize(gray, (img_size, img_size))

            images.append(gray)
            labels.append(cls)

    return np.array(images, dtype=np.float32) / 255.0, np.array(labels)

In [9]:
def apply_filters(images):
    # Sobel kernels (manual NumPy)
    Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
    Ky = Kx.T

    # Laplacian kernel (manual NumPy)
    Kl = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)

    filtered = []
    for img in images:
        img8 = (img * 255).astype(np.uint8)
        sx   = cv2.filter2D(img8, -1, Kx).astype(np.float32) / 255.0
        sy   = cv2.filter2D(img8, -1, Ky).astype(np.float32) / 255.0
        lap  = cv2.filter2D(img8, -1, Kl).astype(np.float32) / 255.0
        sobel = np.sqrt(sx**2 + sy**2)
        # Stack: original + sobel magnitude + laplacian → 3 channels
        stacked = np.stack([img, sobel, lap], axis=-1)
        filtered.append(stacked)

    return np.array(filtered, dtype=np.float32)

In [14]:
# Unzip the data if the folder doesn't exist
if not os.path.exists(DATA_DIR):
    import zipfile
    print("Unzipping data...")
    # Using the path from cell pFC2WsKxy9dj
    zip_path = '/content/drive/MyDrive/archive.zip'
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")

print("Loading images...")
images_gray, labels_raw = load_images(DATA_DIR)

le      = LabelEncoder()
labels  = le.fit_transform(labels_raw)
n_cls   = len(le.classes_)
print(f"Classes: {n_cls}  |  Samples: {len(labels)}")

# 3-channel filtered images for CNN / MobileNet
images_3ch = apply_filters(images_gray)   # (N, 128, 128, 3)

# Flat vectors for NN baseline
images_flat = images_gray.reshape(len(images_gray), -1)  # (N, 128*128)

X_flat_tr, X_flat_te, y_tr, y_te = train_test_split(images_flat, labels, test_size=0.2, random_state=42)
# Fixed double split to avoid redundant processing
X_3ch_tr, X_3ch_te, _, _ = train_test_split(images_3ch, labels, test_size=0.2, random_state=42)

Loading images...
Classes: 15  |  Samples: 20638


In [10]:
print("\n── Flat NN baseline ──")
flat_nn = tf.keras.models.Sequential([
    layers.Input(shape=(IMG_SIZE * IMG_SIZE,)),
    layers.Dense(512, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(n_cls, activation="softmax"),
], name="FlatNN")

flat_nn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
flat_nn.fit(X_flat_tr, y_tr, validation_data=(X_flat_te, y_te),
            epochs=EPOCHS_NN, batch_size=BATCH, verbose=1)
_, nn_acc = flat_nn.evaluate(X_flat_te, y_te, verbose=0)
print(f"Flat NN Test Accuracy: {nn_acc:.4f}")


── Flat NN baseline ──


NameError: name 'n_cls' is not defined

In [11]:
print("\n── CNN from scratch ──")
cnn = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(n_cls, activation="softmax"),
], name="CNN_Scratch")

cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(X_3ch_tr, y_tr, validation_data=(X_3ch_te, y_te),
        epochs=EPOCHS_CNN, batch_size=BATCH, verbose=1)
_, cnn_acc = cnn.evaluate(X_3ch_te, y_te, verbose=0)
print(f"CNN Scratch Test Accuracy: {cnn_acc:.4f}")
print(f"\nComparison → Flat NN: {nn_acc:.4f}  |  CNN: {cnn_acc:.4f}")


── CNN from scratch ──


NameError: name 'n_cls' is not defined

In [12]:
print("\n── MobileNetV2 fine-tune ──")

# Resize to 96 for speed (MobileNetV2 min = 96)
MN_SIZE = 96
def resize_batch(arr, size):
    return np.array([cv2.resize(img, (size, size)) for img in arr])

X_mn_tr = resize_batch(X_3ch_tr, MN_SIZE)
X_mn_te = resize_batch(X_3ch_te, MN_SIZE)

base = applications.MobileNetV2(input_shape=(MN_SIZE, MN_SIZE, 3),
                                include_top=False, weights="imagenet")
base.trainable = False  # freeze base

mn_input = layers.Input(shape=(MN_SIZE, MN_SIZE, 3))
x = applications.mobilenet_v2.preprocess_input(mn_input * 255.0)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(n_cls, activation="softmax")(x)

mn_model = models.Model(mn_input, out, name="MobileNetV2_FineTune")
mn_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Phase 1: train head only
lr_sched = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)
mn_model.fit(X_mn_tr, y_tr, validation_data=(X_mn_te, y_te),
             epochs=EPOCHS_MN, batch_size=BATCH, callbacks=[lr_sched], verbose=1)

# Phase 2: unfreeze top 20 layers with lower LR
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False

mn_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),   # differential LR
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
mn_model.fit(X_mn_tr, y_tr, validation_data=(X_mn_te, y_te),
             epochs=EPOCHS_MN, batch_size=BATCH, callbacks=[lr_sched], verbose=1)

_, mn_acc = mn_model.evaluate(X_mn_te, y_te, verbose=0)
print(f"MobileNetV2 Test Accuracy: {mn_acc:.4f}")


── MobileNetV2 fine-tune ──


NameError: name 'X_3ch_tr' is not defined

In [13]:
print("\n╔══════════════════════════════╗")
print("║      RESULTS SUMMARY         ║")
print("╠══════════════════════════════╣")
print(f"║ Flat NN      : {nn_acc:.4f}         ║")
print(f"║ CNN Scratch  : {cnn_acc:.4f}         ║")
print(f"║ MobileNetV2  : {mn_acc:.4f}         ║")
print("╚══════════════════════════════╝")


╔══════════════════════════════╗
║      RESULTS SUMMARY         ║
╠══════════════════════════════╣


NameError: name 'nn_acc' is not defined